# Import libraries and load cleaned datasets

In [3]:
import pandas as pd

In [4]:
sales = pd.read_csv("../datasets/processed datasets/clean_sales.csv")
products = pd.read_csv("../datasets/processed datasets/clean_products.csv")
stores = pd.read_csv("../datasets/processed datasets/clean_stores.csv")
shelf = pd.read_csv("../datasets/processed datasets/clean_shelf_audit.csv")

# Merge datasets

In [5]:
df = sales.merge(products,on='sku_id')
df = df.merge(stores,on='store_id')

# Revenue per Unit

In [6]:
print(df.columns.tolist())

['transaction_id', 'transaction_date', 'store_id', 'sku_id', 'quantity_sold', 'sales_amount', 'product_name', 'brand', 'category', 'unit_price', 'store_name', 'city', 'state', 'region', 'store_size']


In [7]:
df['revenue_per_unit'] = (
    df['sales_amount'] /
    df['quantity_sold']
)

# High Revenue Product Flag

In [8]:
median_sales = df['sales_amount'].median()

df['high_revenue_product'] = (
    df['sales_amount'] > median_sales
).astype(int)

# Average Sales per SKU

In [9]:
avg_sales = (
    df.groupby('sku_id')['quantity_sold']
      .mean()
      .reset_index()
)

avg_sales.rename(
    columns={'quantity_sold':'avg_quantity_sold'},
    inplace=True
)

df = df.merge(
    avg_sales,
    on='sku_id'
)

# Revenue Category

In [10]:
df['revenue_category'] = pd.qcut(
    df['sales_amount'],
    q=3,
    labels=['Low','Medium','High']
)

# Transaction Date Features

Convert date

In [11]:
df['transaction_date'] = pd.to_datetime(
    df['transaction_date']
)

Convert month

In [12]:
df['month'] = df['transaction_date'].dt.month_name()

Convert Year

In [13]:
df['year'] = df['transaction_date'].dt.year

Convert Quarter

In [14]:
df['quarter'] = df['transaction_date'].dt.quarter

Day Name

In [15]:
df['day_name'] = df['transaction_date'].dt.day_name()

Weekend Flag

In [16]:
df['is_weekend'] = (
    df['day_name']
      .isin(['Saturday','Sunday'])
)

# Revenue Contribution %

In [17]:
total_sales = df['sales_amount'].sum()

df['revenue_contribution_pct'] = (
    df['sales_amount']
    / total_sales
)*100

# Store Size Category

In [19]:
df['store_size'].unique()

<StringArray>
['Small', 'Medium', 'Large']
Length: 3, dtype: str

In [20]:
df['store_size_category'] = df['store_size']

# Region Revenue Rank

In [21]:
region_sales = (
    df.groupby('region')['sales_amount']
      .sum()
      .rank(ascending=False)
)

print(region_sales)

region
East     4.0
North    2.0
South    3.0
West     1.0
Name: sales_amount, dtype: float64


# Save Feature-Enriched Dataset

In [22]:
df.to_csv(
    "../datasets/metadata/feature_engineered_data.csv",
    index=False
)